# Project - Airline AI Assistant with Anthropic

Imagine the model is an airline AI customer support assistant.

In [ ]:
import os

# Load env variables
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv('ANTHROPIC_API_KEY')
if api_key:
    print(f"Anthropic API Key exists and begins {api_key[:8]}")
else:
    print("Anthropic API Key not set")

In [ ]:
from anthropic import Anthropic
import json
import sqlite3

client = Anthropic(api_key=api_key)
# model = "claude-3-5-sonnet-20241022"
model = "claude-haiku-5-5"

## System Message

Define the system prompt for the airline assistant.

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

## Database Setup

Create a SQLite database to store ticket prices.

In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

print("Database initialized")

## Tool Functions

Define the functions that the AI can call.

In [ ]:
def get_ticket_price(city: str) -> str:
    """Get the ticket price for a destination city."""
    print(f"Tool called: Getting price for {city}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
def set_ticket_price(city: str, price: float) -> str:
    """Set the ticket price for a destination city."""
    print(f"Tool called: Setting price for {city} to ${price}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', 
                      (city.lower(), price, price))
        conn.commit()
    return f"Successfully set ticket price to {city} as ${price}"

## Tool Definitions for Anthropic

Define the tools in the format required by the Anthropic API.

In [ ]:
tools = [
    {
        "name": "get_ticket_price",
        "description": "Get the price of a return ticket to the destination city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city that the customer wants to travel to",
                },
            },
            "required": ["city"],
        }
    },
    {
        "name": "set_ticket_price",
        "description": "Set the price of a return ticket for a destination city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city to set the ticket price for",
                },
                "price": {
                    "type": "number",
                    "description": "The price of the ticket in dollars",
                },
            },
            "required": ["city", "price"],
        }
    }
]

print("Tools defined")

## Populate Database

Add some initial ticket prices.

In [ ]:
ticket_prices = {"london": 799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

# Test a retrieval
print(get_ticket_price("Tokyo"))

## Tool Handler

Process tool calls from Claude and return the results.

In [ ]:
def handle_tool_call(tool_name: str, tool_input: dict) -> str:
    """Execute the requested tool and return the result."""
    if tool_name == "get_ticket_price":
        return get_ticket_price(tool_input["city"])
    elif tool_name == "set_ticket_price":
        return set_ticket_price(tool_input["city"], tool_input["price"])
    else:
        return f"Unknown tool: {tool_name}"

## Chat Function with Tool Use

Implement the chat function that handles tool calls.

In [ ]:
def chat(user_message: str, conversation_history: list = None) -> str:
    """Send a message and get a response, handling tool calls."""
    if conversation_history is None:
        conversation_history = []
    
    # Add the user message to history
    conversation_history.append({
        "role": "user",
        "content": user_message
    })
    
    # Keep calling the API until we get a final response (not a tool use)
    while True:
        response = client.messages.create(
            model=model,
            max_tokens=1024,
            system=system_message,
            tools=tools,
            messages=conversation_history
        )
        
        # Check if Claude wants to use a tool
        if response.stop_reason == "tool_use":
            # Find the tool use block
            tool_use_block = None
            for block in response.content:
                if block.type == "tool_use":
                    tool_use_block = block
                    break
            
            if tool_use_block:
                # Execute the tool
                tool_result = handle_tool_call(tool_use_block.name, tool_use_block.input)
                
                # Add Claude's response to history
                conversation_history.append({
                    "role": "assistant",
                    "content": response.content
                })
                
                # Add the tool result to history
                conversation_history.append({
                    "role": "user",
                    "content": [
                        {
                            "type": "tool_result",
                            "tool_use_id": tool_use_block.id,
                            "content": tool_result
                        }
                    ]
                })
        else:
            # Claude has finished and provided a final response
            # Extract the text response
            final_response = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_response += block.text
            
            # Add to history
            conversation_history.append({
                "role": "assistant",
                "content": final_response
            })
            
            return final_response, conversation_history

print("Chat function defined")

## Test the Chat Function

Test some basic interactions.

In [ ]:
# Start a conversation
history = []

# Test 1: Ask for a ticket price
response, history = chat("What is the price for a ticket to London?", history)
print(f"User: What is the price for a ticket to London?")
print(f"Assistant: {response}")
print()

In [ ]:
# Test 2: Ask about multiple cities
response, history = chat("How much are tickets to Paris and Tokyo?", history)
print(f"User: How much are tickets to Paris and Tokyo?")
print(f"Assistant: {response}")
print()

In [ ]:
# Test 3: Set a price
response, history = chat("Please set the ticket price to Berlin to $599.", history)
print(f"User: Please set the ticket price to Berlin to $599.")
print(f"Assistant: {response}")
print()

In [ ]:
# Test 4: Verify the price was set
response, history = chat("What is the price for Berlin now?", history)
print(f"User: What is the price for Berlin now?")
print(f"Assistant: {response}")
print()

## Interactive Chat Interface

For a full interactive experience, uncomment and run the code below.

In [ ]:
# Uncomment the following to use Gradio for an interactive chat interface
# Note: Gradio may not work in all environments (like remote sessions)

# import gradio as gr

# def gradio_chat(message, history):
#     # Convert gradio history format to our internal format
#     internal_history = [{"role": h["role"], "content": h["content"]} for h in history]
#     response, internal_history = chat(message, internal_history)
#     return response

# gr.ChatInterface(fn=gradio_chat, type="messages").launch()


## Exercise

Add a new tool to list all available destinations and their prices!

In [ ]:
# SOLUTION: Add a tool to list all destinations

def list_destinations() -> str:
    """List all available destinations and their prices."""
    print("Tool called: Listing all destinations")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT city, price FROM prices ORDER BY city')
        results = cursor.fetchall()
        if results:
            destinations = ", ".join([f"{city.capitalize()} (${price})" for city, price in results])
            return f"Available destinations: {destinations}"
        else:
            return "No destinations available"

In [ ]:
# Add the list_destinations tool to the tools list
tools.append({
    "name": "list_destinations",
    "description": "List all available destinations and their ticket prices.",
    "input_schema": {
        "type": "object",
        "properties": {},
        "required": [],
    }
})

# Update the handle_tool_call function to include the new tool
def handle_tool_call(tool_name: str, tool_input: dict) -> str:
    """Execute the requested tool and return the result."""
    if tool_name == "get_ticket_price":
        return get_ticket_price(tool_input["city"])
    elif tool_name == "set_ticket_price":
        return set_ticket_price(tool_input["city"], tool_input["price"])
    elif tool_name == "list_destinations":
        return list_destinations()
    else:
        return f"Unknown tool: {tool_name}"

In [ ]:
# Test the new tool
response, history = chat("What destinations do we serve?", history)
print(f"User: What destinations do we serve?")
print(f"Assistant: {response}")
print()

## Key Differences from OpenAI Implementation

1. **Tool Definition Format**: Anthropic uses `input_schema` instead of `parameters`
2. **Stop Reason**: Anthropic uses `stop_reason == "tool_use"` instead of `finish_reason == "tool_calls"`
3. **Tool Result Format**: Anthropic requires a `tool_result` block with `type`, `tool_use_id`, and `content`
4. **Response Structure**: Tool use is a block within `response.content` with `type == "tool_use"`
5. **API Client**: Uses Anthropic client instead of OpenAI client